In [ ]:
!pip install scikit-survival pandas numpy

### Pre-process Data with Fairness

In [ ]:
import pandas as pd
import numpy as np
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored
from sklearn.model_selection import train_test_split

# Load data
data = pd.read_csv('/content/features_encoded_last.csv')

# Prepare survival targets
y = data[['efs', 'efs_time']]
y_structured = np.array([(e, t) for e, t in zip(y['efs'], y['efs_time'])],
                        dtype=[('event', bool), ('time', float)])

# Features: Drop non-predictive columns
X = data.drop(['ID', 'efs', 'efs_time'], axis=1)

# Handle missing values
X.fillna(X.median(), inplace=True)

# Extract race groups (one-hot encoded columns)
race_columns = [col for col in X.columns if col.startswith('race_group_')]
X_race = X[race_columns].idxmax(axis=1).str.replace('race_group_', '')

# Compute fairness weights: Inverse of race group frequency
race_counts = X_race.value_counts()
sample_weights = X_race.map(race_counts).apply(lambda x: 1 / x).values

In [ ]:
# Split data with stratification by race
X_train, X_val, y_train, y_val, weights_train, weights_val = train_test_split(
    X.drop(race_columns, axis=1),  # Remove race columns from features
    y_structured,
    sample_weights,
    test_size=0.2,
    stratify=X_race,  # Ensure balanced race distribution
    random_state=42
)

# Train RSF with fairness weights
rsf = RandomSurvivalForest(
    n_estimators=200,
    max_depth=10,
    min_samples_split=10,
    random_state=42
)
rsf.fit(X_train, y_train, sample_weight=weights_train)

RandomSurvivalForest(max_depth=10, min_samples_split=10, n_estimators=200,
                     random_state=42)

In [ ]:
# Predict risk scores
risk_scores = rsf.predict(X_val)

# Extract race labels for validation set
val_race = X_race.iloc[X_val.index]

# Calculate C-index per race group
c_indices = {}
for race in val_race.unique():
    mask = (val_race == race).values
    if sum(mask) < 2:
        continue
    event = y_val[mask]['event']
    time = y_val[mask]['time']
    c_index = concordance_index_censored(event, time, risk_scores[mask])[0]
    c_indices[race] = c_index

# Compute stratified score (mean - std)
mean_c = np.mean(list(c_indices.values()))
std_c = np.std(list(c_indices.values()))
stratified_score = mean_c - std_c

print(f"Stratified C-index: {stratified_score:.4f}")

Stratified C-index: 0.6281
Per-group C-indices: {'More than one race': 0.6369345114284207, 'Asian': 0.6371293875199433, 'Native Hawaiian or other Pacific Islander': 0.6243446972683916, 'Black or African-American': 0.6305361337008024, 'White': 0.646337031204029, 'American Indian or Alaska Native': 0.6575744098625822}
